# Preprocesamiento — Cancelaciones de reservas hoteleras

- **Proyecto:** Práctica Final · Machine Learning y Deep Learning
- **Máster:** Inteligencia Artificial, Cloud Computing y DevOps · PontIA.tech
- **Fase:** 3 — Preprocesamiento de datos

---

## Objetivo

Implementar y validar el módulo `src/data_loader.py` que prepara los datos para
los modelos. A diferencia del EDA, aquí escribimos código de producción reutilizable
en módulos `.py` y solo usamos este notebook para PROBAR que las funciones funcionan.

## Sub-fases

Este notebook se actualiza incrementalmente a medida que añadimos funcionalidad
al módulo `data_loader.py`:

1. 🔹 Carga del dataset (sub-fase 3.1).
2. 🔹 Limpieza inicial: eliminación de leakage y outliers (sub-fase 3.2).
3. 🔹 Pipeline de transformaciones (sub-fase 3.3).
4. 🔹 Split train/test estratificado (sub-fase 3.4).
5. 🔹 Función orquestadora `preparar_datos()` (sub-fase 3.5).

In [1]:
"""Setup del notebook de preprocesamiento."""

# Configurar el path para poder importar desde src/
import sys
from pathlib import Path

# Subir dos niveles desde notebooks/exploracion/ hasta la raíz del proyecto
PATH_PROYECTO = Path.cwd().parents[1] if "notebooks" in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(PATH_PROYECTO))

# Imports estándar
import pandas as pd
import numpy as np

# Imports de nuestro módulo
from src.data_loader import cargar_dataset, TARGET_COLUMN, SEED, PATH_DATASET

print(f"Raíz del proyecto: {PATH_PROYECTO}")
print(f"Dataset previsto:  {PATH_DATASET}")
print(f"Target:            {TARGET_COLUMN}")
print(f"SEED:              {SEED}")

Raíz del proyecto: c:\Juan\Pontia\ML\practica-final-ml\practica-final-ml
Dataset previsto:  C:\Juan\Pontia\ML\practica-final-ml\practica-final-ml\data\raw\dataset_practica_final.csv
Target:            is_canceled
SEED:              42


## Sub-fase 3.1 — Carga del dataset

Probamos que la función `cargar_dataset()` del módulo carga correctamente los datos.

In [2]:
"""Prueba de la función cargar_dataset()."""

df = cargar_dataset()

print(f"Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Primeras columnas: {list(df.columns[:8])}")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

df.head(3)

Dimensiones: 119,390 filas × 32 columnas
Primeras columnas: ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights']
Memoria: 104.83 MB


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


In [3]:
"""Prueba de manejo de error: ruta inválida."""

from pathlib import Path

try:
    cargar_dataset(Path("/ruta/inventada/no_existe.csv"))
except FileNotFoundError as e:
    print(f"✅ Error capturado correctamente: {e}")

✅ Error capturado correctamente: No se encuentra el dataset en \ruta\inventada\no_existe.csv. Asegúrate de que el archivo está en data/raw/.


## Sub-fase 3.2 — Limpieza inicial

Probamos las funciones de limpieza:

- `eliminar_columnas_leakage()`: quita reservation_status, reservation_status_date, assigned_room_type.
- `eliminar_outliers_imposibles()`: quita filas con adr<0 o adults>10.
- `crear_flags_nulos()`: añade tiene_agent y tiene_company.
- `limpiar_dataset()`: orquesta las tres anteriores.

In [4]:
"""Recargar imports del módulo data_loader (por si se modificó)."""

# Recargar el módulo en caso de haber modificado data_loader.py durante esta sesión
import importlib
from src import data_loader
importlib.reload(data_loader)

from src.data_loader import (
    cargar_dataset,
    eliminar_columnas_leakage,
    eliminar_outliers_imposibles,
    crear_flags_nulos,
    limpiar_dataset,
    COLUMNAS_LEAKAGE,
)

print("Funciones importadas correctamente.")
print(f"Columnas con leakage definidas: {COLUMNAS_LEAKAGE}")

Funciones importadas correctamente.
Columnas con leakage definidas: ['reservation_status', 'reservation_status_date', 'assigned_room_type']


In [5]:
"""Probar eliminar_columnas_leakage()."""

df = cargar_dataset()
print(f"Dimensiones antes: {df.shape}")
print(f"¿reservation_status en columnas?: {'reservation_status' in df.columns}")

df_sin_leakage = eliminar_columnas_leakage(df)
print(f"\nDimensiones después: {df_sin_leakage.shape}")
print(f"¿reservation_status en columnas?: {'reservation_status' in df_sin_leakage.columns}")
print(f"¿assigned_room_type en columnas?: {'assigned_room_type' in df_sin_leakage.columns}")

# Comprobar que el DataFrame original NO se modificó
print(f"\n¿El df original sigue intacto?: {'reservation_status' in df.columns}")

Dimensiones antes: (119390, 32)
¿reservation_status en columnas?: True

Dimensiones después: (119390, 29)
¿reservation_status en columnas?: False
¿assigned_room_type en columnas?: False

¿El df original sigue intacto?: True


In [6]:
"""Probar eliminar_outliers_imposibles()."""

print(f"Filas antes: {len(df):,}")
print(f"  - adr < 0:     {(df['adr'] < 0).sum()} filas")
print(f"  - adults > 10: {(df['adults'] > 10).sum()} filas")

df_sin_outliers = eliminar_outliers_imposibles(df)

print(f"\nFilas después: {len(df_sin_outliers):,}")
print(f"  - adr < 0:     {(df_sin_outliers['adr'] < 0).sum()} filas (debe ser 0)")
print(f"  - adults > 10: {(df_sin_outliers['adults'] > 10).sum()} filas (debe ser 0)")

Filas antes: 119,390
  - adr < 0:     1 filas
  - adults > 10: 12 filas
Eliminadas 13 filas con valores imposibles (0.011% del total).

Filas después: 119,377
  - adr < 0:     0 filas (debe ser 0)
  - adults > 10: 0 filas (debe ser 0)


In [7]:
"""Probar crear_flags_nulos()."""

df_con_flags = crear_flags_nulos(df)

print(f"Columnas nuevas creadas: {[c for c in df_con_flags.columns if c not in df.columns]}")
print(f"\nDistribución de tiene_agent:")
print(df_con_flags["tiene_agent"].value_counts())
print(f"\nDistribución de tiene_company:")
print(df_con_flags["tiene_company"].value_counts())

# Confirmar correspondencia con los nulos
print(f"\nVerificación:")
print(f"  tiene_agent=0 ({(df_con_flags['tiene_agent']==0).sum():,}) vs agent nulos ({df['agent'].isnull().sum():,})")
print(f"  tiene_company=0 ({(df_con_flags['tiene_company']==0).sum():,}) vs company nulos ({df['company'].isnull().sum():,})")

Columnas nuevas creadas: ['tiene_agent', 'tiene_company']

Distribución de tiene_agent:
tiene_agent
1    103050
0     16340
Name: count, dtype: int64

Distribución de tiene_company:
tiene_company
0    112593
1      6797
Name: count, dtype: int64

Verificación:
  tiene_agent=0 (16,340) vs agent nulos (16,340)
  tiene_company=0 (112,593) vs company nulos (112,593)


In [8]:
"""Probar limpiar_dataset() (la función orquestadora)."""

df_original = cargar_dataset()
df_limpio = limpiar_dataset(df_original)

print(f"Dimensiones originales: {df_original.shape}")
print(f"Dimensiones tras limpieza: {df_limpio.shape}")
print(f"\nColumnas eliminadas: {set(df_original.columns) - set(df_limpio.columns)}")
print(f"Columnas añadidas:   {set(df_limpio.columns) - set(df_original.columns)}")

print(f"\nResumen de la limpieza:")
print(f"  - Filas: {df_original.shape[0]:,} → {df_limpio.shape[0]:,}")
print(f"  - Columnas: {df_original.shape[1]} → {df_limpio.shape[1]}")

Eliminadas 13 filas con valores imposibles (0.011% del total).
Dimensiones originales: (119390, 32)
Dimensiones tras limpieza: (119377, 31)

Columnas eliminadas: {'reservation_status', 'reservation_status_date', 'assigned_room_type'}
Columnas añadidas:   {'tiene_agent', 'tiene_company'}

Resumen de la limpieza:
  - Filas: 119,390 → 119,377
  - Columnas: 32 → 31


**Resultados de la sub-fase 3.2:**

- Las 3 funciones de limpieza funcionan correctamente.
- La función orquestadora `limpiar_dataset()` aplica todas en orden.
- Las funciones devuelven copias, no modifican el DataFrame original.
- El dataset limpio pasa de 32 columnas a ~31 (eliminamos 3 con leakage, añadimos 2 flags).
- Se eliminan unas pocas filas con valores imposibles (~3-5 filas).

**Listo para la siguiente sub-fase: pipeline de transformaciones (sub-fase 3.3).**